# Импорт датасета

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("women-clothing-accessories.3-class.balanced.csv", sep="\t")

In [3]:
df

,review,sentiment
0,качество плохое пошив ужасный (горловина напер...,negative
1,"Товар отдали другому человеку, я не получила п...",negative
2,"Ужасная синтетика! Тонкая, ничего общего с пре...",negative
3,"товар не пришел, продавец продлил защиту без м...",negative
4,"Кофточка голая синтетика, носить не возможно.",negative
...,...,...
89995,сделано достаточно хорошо. на ткани сделан рис...,positive
89996,Накидка шикарная. Спасибо большое провдо линяе...,positive
89997,спасибо большое ) продовца рекомендую.. заказа...,positive
89998,Очень довольна заказом! Меньше месяца в РБ. К...,positive


In [4]:
#посчтали 3 класса
df['sentiment'].unique()

array(['negative', 'neautral', 'positive'], dtype=object)

# Выборки

In [5]:
from sklearn.model_selection import train_test_split
#соотношение 70/30, рандом 34 для воспроизводимости
df_train, df_test = train_test_split(df, test_size=0.3, random_state=34, stratify=df['sentiment'])

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
#для ml векторизация
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))


X_train = tfidf.fit_transform(df_train['review'])
X_test = tfidf.transform(df_test['review'])

y_train = df_train['sentiment']
y_test = df_test['sentiment']

# ML модели

In [22]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(max_iter=10000)
lr.fit(X_train, y_train)
lr_predictions = lr.predict(X_test)

In [33]:
from sklearn.metrics import accuracy_score, f1_score
acc = accuracy_score(y_test, lr_predictions)
f1 = f1_score(y_test, lr_predictions, average='weighted')
print("Логистическая регрессия")
print(f'Accuracy: {acc: .3f}')
print(f'F1-score: {f1 : .3f}')

Логистическая регрессия
Accuracy:  0.746
F1-score:  0.748


In [23]:
from sklearn.naive_bayes import MultinomialNB
nb = MultinomialNB()
nb.fit(X_train, y_train)
nb_predictions = nb.predict(X_test)

In [34]:
acc = accuracy_score(y_test, nb_predictions)
f1 = f1_score(y_test, nb_predictions, average='weighted')
print("Наивный Байес")
print(f'Accuracy: {acc: .3f}')
print(f'F1-score: {f1 : .3f}')

Наивный Байес
Accuracy:  0.727
F1-score:  0.728


In [27]:
from catboost import CatBoostClassifier
cb = CatBoostClassifier(iterations=1000, loss_function='MultiClass', verbose=100)
cb.fit(X_train, y_train)
cb_predictions = cb.predict(X_test)

Learning rate set to 0.097744
0:	learn: 1.0660043	total: 269ms	remaining: 4m 29s
100:	learn: 0.7395772	total: 23.1s	remaining: 3m 25s
200:	learn: 0.6820733	total: 43.7s	remaining: 2m 53s
300:	learn: 0.6512265	total: 1m 3s	remaining: 2m 28s
400:	learn: 0.6316189	total: 1m 24s	remaining: 2m 5s
500:	learn: 0.6168900	total: 1m 48s	remaining: 1m 47s
600:	learn: 0.6061996	total: 2m 8s	remaining: 1m 24s
700:	learn: 0.5969618	total: 2m 28s	remaining: 1m 3s
800:	learn: 0.5896149	total: 2m 47s	remaining: 41.7s
900:	learn: 0.5831568	total: 3m 7s	remaining: 20.6s
999:	learn: 0.5773709	total: 3m 27s	remaining: 0us


In [35]:
acc = accuracy_score(y_test, cb_predictions)
f1 = f1_score(y_test, cb_predictions, average='weighted')
print("Деревья")
print(f'Accuracy: {acc: .3f}')
print(f'F1-score: {f1 : .3f}')

Деревья
Accuracy:  0.734
F1-score:  0.736


# NLP

In [1]:
from deeppavlov import build_model
RUmodel = build_model('rusentiment_bert', download=False, install=False)

C:\Users\ч\AppData\Local\Programs\Python\Python39\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\ч\AppData\Local\Programs\Python\Python39\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at bert-base-multilingual-cased were not used when initializing BertForSequenceClassification: ['cls.predictions.bias', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.weight']
- This IS expected if you a

In [7]:
#ограничиваемся 1000 строк для экономии времени
df_nlp = df_test.sample(1000)

In [11]:

df_nlp['byModel'] = df_nlp['review'].apply(lambda x: RUmodel([x])[0])

In [12]:
df_nlp

,review,sentiment,byModel
35710,не пришло.,neautral,neutral
10928,"посылку получила через месяц, куртка вся в пят...",negative,neutral
27719,"Пришел размер маленький xl,и качество костюма ...",negative,neutral
84667,thank you very much!!!,positive,positive
21790,"Они очень огромные, на очень большого человека",negative,neutral
...,...,...,...
22968,пришли с дыркой.,negative,neutral
16589,"Качество так себе, красная цена на барахолке р...",negative,positive
28727,Товар не пришёл. Деньги не вернули.,negative,neutral
48045,цвет белый обоссаный на грудь рассчитано разме...,neautral,neutral


In [20]:
mapp = {
 "negative" : "negative",
    "neutral" : "neautral",
    "positive" : "positive",
    "skip" : "neautral",
    "speech" : "positive"
}
df_nlp['byModel'] = df_nlp['byModel'].apply(lambda x: mapp[x])

In [21]:
df_nlp

,review,sentiment,byModel
35710,не пришло.,neautral,neautral
10928,"посылку получила через месяц, куртка вся в пят...",negative,neautral
27719,"Пришел размер маленький xl,и качество костюма ...",negative,neautral
84667,thank you very much!!!,positive,positive
21790,"Они очень огромные, на очень большого человека",negative,neautral
...,...,...,...
22968,пришли с дыркой.,negative,neautral
16589,"Качество так себе, красная цена на барахолке р...",negative,positive
28727,Товар не пришёл. Деньги не вернули.,negative,neautral
48045,цвет белый обоссаный на грудь рассчитано разме...,neautral,neautral


In [26]:
from sklearn.metrics import accuracy_score, f1_score
acc = accuracy_score(df_nlp["sentiment"], df_nlp["byModel"])
f1 = f1_score(df_nlp["sentiment"], df_nlp["byModel"], average='weighted')
print("BERT")
print(f'Accuracy: {acc: .3f}')
print(f'F1-score: {f1 : .3f}')

BERT
Accuracy:  0.509
F1-score:  0.499


# Вывод

In [8]:
average_w = df_nlp["review"].apply(lambda x: len(x.split(" "))).mean()
average_w

18.494

Наибольшая метрика качества у логистической регрессии. Можно предположить, что для данных целей лучше использовать классические ML модели, так как у Bert accuracy на уровне 0.5 (случайное угадывание aka попал-не попал). Данное явление объяснимо, так как rusentiment использует 5 классов вместо 3, плюс среднее количество слов в отзыве (18) может быть недостаточным для правильного предсказания